# Standalone embedding ablation — AIC L01–L25

Một notebook duy nhất, không gọi notebook/backend khác và không dùng OpenAI/Groq. Dùng image embeddings có sẵn trên Zilliz cho OpenCLIP, SigLIP2, Qwen3-VL; tạo query nhỏ bằng NLLB local; xuất kết quả vào Kaggle Output.

Yêu cầu: GPU + Internet, attach dataset `thnhlcdng/ablation`, Kaggle Secrets `MILVUS_URI` và `MILVUS_TOKEN`.

In [ ]:
# ===== CONFIG + INSTALL =====
import os, re, gc, json, math, time, sqlite3, shutil, subprocess, sys
from pathlib import Path

DB_PATH = Path('/kaggle/input/datasets/thnhlcdng/ablation/dev_search_local.db')
REPO_URL = 'https://github.com/zintomvn/Multimodal-Retrieval.git'
BRANCH = 'experiment/embedding-ablation-l01-l25'
REPO = Path('/kaggle/working/ablation-source')
OUTPUT = Path('/kaggle/working/embedding_ablation_l01_l25_results')
TOP_K, SEARCH_LIMIT = 100, 1000
PERSPECTIVE_COUNTS = [3, 5, 7]
BATCH_MIN, BATCH_MAX = 1, 25

if not DB_PATH.is_file():
    raise FileNotFoundError(f'Missing database: {DB_PATH}')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pymilvus>=2.5,<2.7','open_clip_torch>=2.32','sentence-transformers>=5.4','transformers>=4.57.3','qwen-vl-utils>=0.0.14','sentencepiece','accelerate'], check=True)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
MILVUS_URI = secrets.get_secret('MILVUS_URI')
MILVUS_TOKEN = secrets.get_secret('MILVUS_TOKEN')
if not MILVUS_URI or not MILVUS_TOKEN:
    raise RuntimeError('Missing Kaggle Secrets MILVUS_URI/MILVUS_TOKEN')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('DB:', DB_PATH)
print('Output:', OUTPUT)


In [ ]:
# ===== LOAD L01-L25 GROUND TRUTH + FRAME MAP; CREATE 7 FROZEN ENGLISH VIEWS LOCALLY =====
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

GT_PATH = REPO / 'data/experiments/aic_2026_groundtruth/aic2026_all_confirmed.csv'
gt = pd.read_csv(GT_PATH, encoding='utf-8-sig')
def pick(row, *names):
    for name in names:
        if name in row.index and pd.notna(row[name]): return str(row[name]).strip()
    return ''
def batch_no(video_id):
    m = re.match(r'L(\d+)_', str(video_id), re.I); return int(m.group(1)) if m else 999
def parse_frames(raw):
    return tuple(int(x) for x in re.findall(r'\d+', str(raw)))
queries = []
for _, row in gt.iterrows():
    qid = pick(row, 'Query ID', 'Original Query ID')
    text = pick(row, 'Query Text', 'Query')
    video = pick(row, 'GT Video ID')
    frames = parse_frames(pick(row, 'GT Frame ID(s)', 'GT Frame ID'))
    status = pick(row, 'Status')
    task = pick(row, 'Task Type')
    if 'kis' not in qid.lower() or not text or not video or not frames: continue
    if status and status.upper() != 'OK': continue
    if task and task.upper() != 'KIS': continue
    if BATCH_MIN <= batch_no(video) <= BATCH_MAX:
        queries.append({'query_id': qid, 'text': text, 'video_id': video, 'frame_ids': frames})
if not queries: raise RuntimeError('No confirmed L01-L25 KIS queries found')

con = sqlite3.connect(DB_PATH)
frame_lookup = {}
for keyframe_id, video_id, frame_idx in con.execute('SELECT keyframe_id, video_id, frame_idx FROM keyframes WHERE is_media_present=1'):
    if BATCH_MIN <= batch_no(video_id) <= BATCH_MAX:
        frame_lookup[str(keyframe_id)] = (str(video_id), int(frame_idx))
con.close()
print('Queries:', len(queries), '| L01-L25 keyframes:', len(frame_lookup))

def source_views(text, n=7):
    text = re.sub(r'\s+', ' ', text).strip()
    parts = [p.strip(' ,.;:-') for p in re.split(r'[.!?;:]|\b(?:sau đó|tiếp theo|cuối cùng|đồng thời|trong khi|bên cạnh|phía trước|phía sau)\b', text, flags=re.I) if p.strip(' ,.;:-')]
    words = text.split(); candidates = list(parts)
    for windows in (n, n-1, n+1):
        for i in range(windows):
            start, end = i*len(words)//windows, (i+1)*len(words)//windows
            if end > start: candidates.append(' '.join(words[start:end]))
    out=[]; seen=set()
    for item in candidates:
        key=item.casefold()
        if item and key not in seen: out.append(item); seen.add(key)
        if len(out)==n: return out
    raise ValueError('Query too short to create seven distinct views')

source_by_query = {q['query_id']: source_views(q['text']) for q in queries}
full_source = {q['query_id']: q['text'] for q in queries}
all_source = list(dict.fromkeys([*full_source.values(), *(x for views in source_by_query.values() for x in views)]))
translator_id = 'facebook/nllb-200-distilled-600M'
tok = AutoTokenizer.from_pretrained(translator_id, src_lang='vie_Latn')
translator = AutoModelForSeq2SeqLM.from_pretrained(translator_id, torch_dtype=torch.float16).to('cuda').eval()
translations = {}
for start in range(0, len(all_source), 16):
    batch = all_source[start:start+16]
    inputs = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to('cuda')
    with torch.inference_mode():
        generated = translator.generate(**inputs, forced_bos_token_id=tok.convert_tokens_to_ids('eng_Latn'), max_new_tokens=128)
    for src, dst in zip(batch, tok.batch_decode(generated, skip_special_tokens=True)):
        translations[src] = re.sub(r'\s+', ' ', dst).strip()
full_queries = {qid: translations[text] for qid, text in full_source.items()}
perspectives = {qid: [translations[x] for x in views] for qid, views in source_by_query.items()}
(OUTPUT/'full_queries.json').write_text(json.dumps(full_queries, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT/'perspectives.json').write_text(json.dumps(perspectives, ensure_ascii=False, indent=2), encoding='utf-8')
del translator, tok; gc.collect(); torch.cuda.empty_cache()
print('Frozen perspectives saved:', OUTPUT/'perspectives.json')
print(queries[0]['query_id'], perspectives[queries[0]['query_id']])


In [ ]:
# ===== DIRECT TEXT EMBEDDING + ZILLIZ SEARCH FOR ALL 3 MODELS =====
import numpy as np
from pymilvus import MilvusClient

SPECS = {
 'openclip': {'collection':'keyframe_embeddings_clip_vith14_quickgelu_dfn5b_v2','dim':1024,'kind':'open_clip','model':'ViT-H-14-quickgelu','pretrained':'dfn5b'},
 'siglip2': {'collection':'keyframe_embeddings_siglip2_so400m16_384_webli_openclip_1152_v1','dim':1152,'kind':'open_clip','model':'ViT-SO400M-16-SigLIP2-384','pretrained':'webli'},
 'qwen3_vl': {'collection':'keyframe_embeddings_qwen3_vl_embedding_2b_2048_v1','dim':2048,'kind':'sentence_transformer','model':'Qwen/Qwen3-VL-Embedding-2B'},
}
client = MilvusClient(uri=MILVUS_URI, token=MILVUS_TOKEN, timeout=120)
for name,spec in SPECS.items():
    if not client.has_collection(collection_name=spec['collection']): raise RuntimeError(f'Missing collection: {spec["collection"]}')
    desc=client.describe_collection(collection_name=spec['collection'])
    vf=next(f for f in desc.get('fields',[]) if f.get('name')=='vector'); dim=int((vf.get('params') or {}).get('dim') or vf.get('dim') or 0)
    if dim!=spec['dim']: raise RuntimeError(f'{name}: expected dim {spec["dim"]}, got {dim}')
    rows=int(client.get_collection_stats(collection_name=spec['collection']).get('row_count') or 0)
    print(name, spec['collection'], rows)
qwen_collection=SPECS['qwen3_vl']['collection']; target_ids=list(frame_lookup); covered=0
for start in range(0,len(target_ids),1000): covered += len(client.get(collection_name=qwen_collection,ids=target_ids[start:start+1000],output_fields=['id']))
print('Qwen L01-L25 coverage:',covered,'/',len(target_ids))
if covered!=len(target_ids): raise RuntimeError('Qwen collection does not fully cover L01-L25')
SCOPE_FILTER=' or '.join(f'video_id like "L{i:02d}_%"' for i in range(BATCH_MIN,BATCH_MAX+1))

def resolve_hit(hit):
    entity = hit.get('entity') or {}
    for key in ('canonical_keyframe_id','mapped_keyframe_id','keyframe_id','frame_id'):
        candidate = str(entity.get(key) or '')
        if candidate in frame_lookup: return candidate
    candidate = str(hit.get('id') or '')
    return candidate if candidate in frame_lookup else None
def gt_rank(ranked_ids, query):
    for rank,kid in enumerate(ranked_ids,1):
        video,frame = frame_lookup[kid]
        if video==query['video_id'] and frame in query['frame_ids']: return rank
    return -1
def search_views(collection, vectors):
    data=np.asarray(vectors,dtype='float32').tolist(); fields=['frame_id','video_id','keyframe_id','canonical_keyframe_id','mapped_keyframe_id']
    try:
        raw=client.search(collection_name=collection,data=data,limit=TOP_K,filter=SCOPE_FILTER,output_fields=fields,search_params={'metric_type':'COSINE'},timeout=120)
        if any(len(hits)<TOP_K for hits in raw): raise RuntimeError('scope filter returned too few hits')
    except Exception as exc:
        print('Milvus scope-filter fallback:',type(exc).__name__)
        raw=client.search(collection_name=collection,data=data,limit=SEARCH_LIMIT,output_fields=fields,search_params={'metric_type':'COSINE'},timeout=120)
    scores={}
    for hits in raw:
        for hit in hits:
            kid=resolve_hit(hit)
            if kid: scores[kid]=max(scores.get(kid,-1e9),float(hit.get('distance',hit.get('score',0))))
    return [kid for kid,_ in sorted(scores.items(),key=lambda x:x[1],reverse=True)[:TOP_K]]

records=[]
for model_name,spec in SPECS.items():
    print('\n===',model_name,'===')
    if spec['kind']=='open_clip':
        import open_clip
        model,_,_=open_clip.create_model_and_transforms(spec['model'],pretrained=spec['pretrained'],device='cuda')
        tokenizer=open_clip.get_tokenizer(spec['model']); model.eval()
        def encode(texts):
            tokens=tokenizer(texts).to('cuda')
            with torch.inference_mode(), torch.autocast('cuda',dtype=torch.float16): vec=model.encode_text(tokens)
            return torch.nn.functional.normalize(vec.float(),dim=-1).cpu().numpy()
    else:
        from sentence_transformers import SentenceTransformer
        model=SentenceTransformer(spec['model'],device='cuda',trust_remote_code=True,model_kwargs={'torch_dtype':torch.float16,'attn_implementation':'sdpa'})
        tokenizer=None
        def encode(texts): return model.encode(texts,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)
    probe=encode(['a test image query'])
    if probe.shape[1]!=spec['dim'] or not np.isfinite(probe).all(): raise RuntimeError(f'{model_name} bad text vectors: {probe.shape}')
    configurations=[('full_query',1)]+[('perspective',n) for n in PERSPECTIVE_COUNTS]
    for qi,q in enumerate(queries,1):
        for mode,n in configurations:
            texts=[full_queries[q['query_id']]] if mode=='full_query' else perspectives[q['query_id']][:n]
            started=time.perf_counter(); vectors=encode(texts); ranked=search_views(spec['collection'],vectors); latency=(time.perf_counter()-started)*1000
            records.append({'model':model_name,'query_mode':mode,'n':n,'query_id':q['query_id'],'gt_video_id':q['video_id'],'gt_frame_ids':';'.join(map(str,q['frame_ids'])),'rank':gt_rank(ranked,q),'latency_ms':round(latency,3),'returned':len(ranked)})
        print(f'{model_name}: {qi}/{len(queries)}')
    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()

per_query=pd.DataFrame(records)
per_query.to_csv(OUTPUT/'per_query.csv',index=False,encoding='utf-8-sig')
print('Search complete:',len(per_query),'measurements')


In [ ]:
# ===== METRICS, PAPER-LIKE RANK TABLE, KAGGLE OUTPUT ZIP =====
def pct(values,q): return float(np.percentile(values,q))
summary=[]
for (model,mode,n),g in per_query.groupby(['model','query_mode','n'],sort=False):
    ranks=g['rank'].astype(int).tolist(); lat=g['latency_ms'].astype(float).tolist()
    row={'model':model,'query_mode':mode,'n':n,'queries':len(g),'MRR':np.mean([1/r if r>0 else 0 for r in ranks]),'mean_latency_ms':np.mean(lat),'p50_latency_ms':pct(lat,50),'p95_latency_ms':pct(lat,95),'misses_at_100':sum(r<0 for r in ranks)}
    for k in (1,5,10,50,100): row[f'Recall@{k}']=np.mean([0<r<=k for r in ranks])
    summary.append(row)
summary=pd.DataFrame(summary); summary.to_csv(OUTPUT/'summary.csv',index=False,encoding='utf-8-sig')
aliases={q['query_id']:f'q{i}' for i,q in enumerate(queries,1)}
rank_table=per_query.pivot_table(index=['model','query_mode','n'],columns='query_id',values='rank',aggfunc='first').reset_index()
rank_table=rank_table.rename(columns=aliases); rank_table.to_csv(OUTPUT/'rank_table.csv',index=False,encoding='utf-8-sig')
lines=['| '+ ' | '.join(map(str,rank_table.columns))+' |','|'+'|'.join(['---']*len(rank_table.columns))+'|']
for _,row in rank_table.iterrows(): lines.append('| '+' | '.join(str(x) for x in row.tolist())+' |')
lines += ['', 'Query mapping:']+[f'- {alias}: `{qid}`' for qid,alias in aliases.items()]
(OUTPUT/'rank_table.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
(OUTPUT/'run_config.json').write_text(json.dumps({'scope':'L01-L25','top_k':TOP_K,'search_limit':SEARCH_LIMIT,'models':SPECS,'perspective_counts':PERSPECTIVE_COUNTS,'query_count':len(queries),'perspective_generator':'facebook/nllb-200-distilled-600M deterministic contiguous chunks'},ensure_ascii=False,indent=2),encoding='utf-8')
archive=shutil.make_archive('/kaggle/working/embedding_ablation_l01_l25_results','zip',OUTPUT)
display(summary)
print('DONE — Kaggle Output directory:',OUTPUT)
print('ZIP:',archive)
